In [1]:
import numpy as np
import pandas as pd

## Det rigtige spørgsmål

Hvad er jeres case-mål?

Typisk vil det være én af disse:

🔹 Forudsige aflysning?

🔹 Optimere stueplanlægning?

🔹 Forudsige overskridelser?

🔹 Kapacitetsudnyttelse?

🔹 Delay prediction?

Det afgør hvad I skal bruge.

✅ Completed

→ Indeholder:

Patient alder

Speciale

Akut / planlagt

Faktiske tider

Forsinkelser

Ressourcer

Personale

❌ Cancelled

→ Indeholder kun:

Planlægningsinfo

Forventet varighed

Procedure

Stue

Dato

Import data - sep using ; 

In [2]:
df_complete = pd.read_csv(
    "../Data and descriptions/Case Rigshospitalet - Completed operations.csv",
    sep=";"
)

df_cancelled = pd.read_csv(
    "../Data and descriptions/Case Rigshospitalet - Cancelled operations.csv",
    sep=";"
)

/var/folders/t7/k_81cy8x6zg28rjf0wl09fl00000gn/T/ipykernel_78307/711997202.py:1: DtypeWarning: Columns (13,14,21,22,23,24,25,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df_complete = pd.read_csv(


check for missing data

In [ ]:

print("shape of completed operations:", df_complete.shape)
print("shape of cancelled operations:", df_cancelled.shape)

#check for missing values
print("Missing values in completed operations:")
print(df_complete.isnull().sum())
print("Missing values in cancelled operations:")
print(df_cancelled.isnull().sum())


Lav dataframe

In [7]:
df_cancelled['Dato og tid'] = pd.to_datetime(df_cancelled['Dato og tid'], format='%Y-%m-%d %H:%M:%S,%f')
df_cancelled['Dato og tid']

df_complete['Dato'] = pd.to_datetime(df_complete['Dato'], format='%Y-%m-%d %H:%M:%S,%f')
df_complete['Dato']



0        2024-03-07
1        2024-01-05
2        2024-01-05
3        2024-01-26
4        2024-10-01
            ...    
133153   2025-12-18
133154   2025-12-23
133155   2025-12-19
133156   2025-12-22
133157   2025-12-19
Name: Dato, Length: 133158, dtype: datetime64[ns]

In [8]:
#print min og max for date for de to dataframes 
print("Cancelled operations - min date:", df_cancelled['Dato og tid'].min())
print("Cancelled operations - max date:", df_cancelled['Dato og tid'].max())
print("Completed operations - min date:", df_complete['Dato'].min())
print("Completed operations - max date:", df_complete['Dato'].max())

Cancelled operations - min date: 2024-01-03 08:05:00
Cancelled operations - max date: 2025-12-30 08:15:00
Completed operations - min date: 2024-01-01 00:00:00
Completed operations - max date: 2025-12-31 00:00:00


1. Hvorfor completed har 247 kolonner
De mange Ressource.* og Staff.* kolonner er i praksis:

One-hot encodede indikatorer (0/1)
for om en bestemt ressource eller personale-type blev brugt.
90%+ af værdierne er 0 / NaN

Det betyder:

Datasættet er meget sparse

Shape siger ikke så meget om reel informationsmængde

2. Cancelled dataset

Cancelled indeholder kun:

Planlægningsinfo

Forventet varighed

Procedure

Stue

Aflysningsårsag (men 467/575 mangler)

Det betyder:

👉 I har næsten ingen strukturel info om aflysninger
👉 I har kun planlægningsniveau

Det er faktisk meget realistisk i hospitalsdata.

In [ ]:
#check for duplicates
print("Duplicate rows in completed operations:", df_complete.duplicated().sum())
print("Duplicate rows in cancelled operations:", df_cancelled.duplicated().sum())
#check for outliers
print("Summary statistics for completed operations:")
print(df_complete.describe())
print("Summary statistics for cancelled operations:")
print(df_cancelled.describe())

Duplicate rows in completed operations: 0
Duplicate rows in cancelled operations: 0
Summary statistics for completed operations:
       Case-ID Anonymous  Patient Alder  Operationsgang ID  \
count      133158.000000  133158.000000      133158.000000   
mean        60156.037437      53.788244       40663.550909   
std         34927.061620      24.889920       52262.972747   
min             1.000000       0.000000         618.000000   
25%         29928.250000      34.000000         622.000000   
50%         60097.500000      61.000000         625.000000   
75%         90338.750000      74.000000      107621.000000   
max        120868.000000     107.000000      133218.000000   

       Forsinkelse (minutter)  Overskredet (minutter)  Staff.Anæstesiolog  \
count           132491.000000           132537.000000             74832.0   
mean                20.511642               25.346198                 1.0   
std                 50.977296               68.894706                 0.0   
min 

Visualize data

## definerer nyt datasæt

In [ ]:
core_cols = [
    "Case-ID Anonymous",
    "Patient Alder",
    "Speciale",
    "Stue",
    "Operationsgang ID",
    "Akut case (J/N)",
    "Dato",
    "Procedure start",
    "Procedure slut",
    "Forsinkelse (minutter)",
    "Overskredet (minutter)"
]

df_core = df_complete[core_cols].copy()

## definer target variable, for om der er cancelled

In [ ]:
df_complete["Cancelled"] = 0
df_cancelled["Cancelled"] = 1
common_cols = list(set(df_complete.columns).intersection(df_cancelled.columns))

df_all = pd.concat([
    df_complete[common_cols],
    df_cancelled[common_cols]
])

print("Shape of combined dataset:", df_all.shape)

Shape of combined dataset: (133733, 4)


Men kun 575 cancelled.

👉 Det er meget ubalanceret (0.4% cancelled)

Det bliver en modelling-udfordring.